# Complete Transit Matrix and Adjusted All-Mode Matrix

Executes `TRANSIT_DEMAND_PLAN.md` now that the train matrix is available:

1. **Train OD 6:00–9:00** from `Input/Matrices/Train_mtx_table.csv` (2019 smartcard data, station-to-station by the station's TAZ; hours 6+7+8 summed; the `TAZ 9999 = שאר התחנות` "other/rest of stations" rows are ignored per instruction).
2. **Complete transit matrix** = filtered RavKav×OnBoard bus + train, at the aggregated areas.
3. **Adjusted all-mode matrix**: `ALL_adjusted = (matrix_avg_ALL − matrix_avg_TRANSIT − matrix_avg_RAIL) + bus + train` — survey keeps CAR/OTHER, measured ticketing replaces the survey's transit components.
4. **Mode-share table** per area and for the LRT corridor.

**Vintage mix (documented, not adjusted): CAR/OTHER base = THS 2018; bus = RavKav 2022; train = smartcards 2019.**

In [1]:
import numpy as np
import pandas as pd
import os

HOURS = ['6', '7', '8']
tr = pd.read_csv('Input/Matrices/Train_mtx_table.csv', encoding='windows-1255')
other_rows = (tr['OriginTAZ'] == 9999) | (tr['DestTAZ'] == 9999)
print(f"rows: {len(tr)} | 'other' (TAZ 9999) rows ignored: {other_rows.sum()} "
      f"({tr.loc[other_rows, HOURS].sum().sum():,.0f} trips 6–9 to/from stations outside the area)")
tr = tr[~other_rows].copy()
tr['trips_6_9'] = tr[HOURS].sum(axis=1)

train_taz = tr.pivot_table(index='OriginTAZ', columns='DestTAZ', values='trips_6_9', aggfunc='sum').fillna(0)
train_taz.index.name = 'OriginTAZ'
os.makedirs('Output/train', exist_ok=True)
train_taz.to_csv('Output/train/train_od_taz_6_9.csv', float_format='%.6g')
print(f"train OD 6–9 (station TAZs): {train_taz.shape}, {train_taz.values.sum():,.0f} avg-day trips (2019 smartcards)")

rows: 380 | 'other' (TAZ 9999) rows ignored: 38 (13,624 trips 6–9 to/from stations outside the area)
train OD 6–9 (station TAZs): (19, 19), 5,535 avg-day trips (2019 smartcards)


## Aggregation to areas and the complete transit matrix

In [2]:
sub_key = pd.read_excel('Input/Submatrix_tazs.xlsx')
taz_to_area = sub_key.set_index('TAZ')['AggAreaCode']
legend = sub_key.drop_duplicates('AggAreaCode').set_index('AggAreaCode')['AggAreaName']

def to_area(m, areas):
    m = m.copy()
    m.index.name, m.columns.name = 'o', 'd'
    long = m.stack().reset_index()
    long.columns = ['o', 'd', 'v']
    long['O'] = long['o'].map(taz_to_area)
    long['D'] = long['d'].map(taz_to_area)
    return (long.dropna(subset=['O', 'D']).groupby(['O', 'D'])['v'].sum()
            .unstack().reindex(index=areas, columns=areas, fill_value=0).fillna(0))

bus_area = pd.read_csv('Output/bus/bus_od_area_new_filtered.csv', index_col=0)
bus_area.columns = bus_area.columns.astype(int)
AREAS = list(bus_area.index)  # the 26 noise-filtered areas

train_area = to_area(train_taz, AREAS)
train_area.index.name = 'AggAreaCode'
train_area.to_csv('Output/train/train_od_area.csv', float_format='%.6g')
print(f"train trips with both ends in the sub-area: {train_area.values.sum():,.0f} "
      f"(stations in sub-area TAZs only; most rail trips cross the boundary)")

os.makedirs('Output/transit', exist_ok=True)
transit_area = bus_area + train_area
transit_area.to_csv('Output/transit/transit_od_area.csv', float_format='%.6g')
print(f"complete transit matrix: {transit_area.shape}, {transit_area.values.sum():,.0f} passengers "
      f"(bus {bus_area.values.sum():,.0f} + train {train_area.values.sum():,.0f})")

train trips with both ends in the sub-area: 962 (stations in sub-area TAZs only; most rail trips cross the boundary)
complete transit matrix: (26, 26), 42,234 passengers (bus 41,272 + train 962)


## Adjusted all-mode matrix (substitution) and mode shares

In [3]:
def load_area(name):
    m = pd.read_csv(f'Output/ths2017/study_taz/submatrices/{name}', index_col=0)
    m.index = m.index.astype(int)
    m.columns = m.columns.astype(int)
    return m.reindex(index=AREAS, columns=AREAS, fill_value=0)

ths_all = load_area('matrix_avg_ALL_area.csv')
ths_transit = load_area('matrix_avg_TRANSIT_area.csv')
ths_rail = load_area('matrix_avg_RAIL_area.csv')

base = ths_all - ths_transit - ths_rail          # survey CAR + OTHER, cell-wise (exact: modes sum to ALL)
assert (base.values >= -1e-6).all()
all_adjusted = base + transit_area
all_adjusted.index.name = 'AggAreaCode'
all_adjusted.to_csv('Output/transit/all_adjusted_area.csv', float_format='%.6g')
print(f"ALL_adjusted: {all_adjusted.values.sum():,.0f} trips "
      f"(survey CAR+OTHER {base.values.sum():,.0f} + measured transit {transit_area.values.sum():,.0f})")
print(f"for reference, unadjusted survey ALL: {ths_all.values.sum():,.0f} "
      f"(survey transit share was {(ths_transit.values.sum() + ths_rail.values.sum()) / ths_all.values.sum():.1%}; "
      f"adjusted transit share: {transit_area.values.sum() / all_adjusted.values.sum():.1%})")

ALL_adjusted: 286,384 trips (survey CAR+OTHER 244,150 + measured transit 42,234)
for reference, unadjusted survey ALL: 270,529 (survey transit share was 9.8%; adjusted transit share: 14.7%)


In [4]:
leg_full = pd.read_csv('Output/ths2017/study_taz/submatrices/area_legend.csv').set_index('AggAreaCode')
share = pd.DataFrame({
    'AggAreaName': legend.reindex(AREAS).values,
    'IsLRT_Corridor': leg_full['IsLRT_Corridor'].reindex(AREAS).values,
    'car_other': base.sum(axis=1).round(0),
    'bus': bus_area.sum(axis=1).round(0),
    'train': train_area.sum(axis=1).round(0),
}, index=AREAS)
share['total'] = share[['car_other', 'bus', 'train']].sum(axis=1)
share['transit_share'] = ((share['bus'] + share['train']) / share['total'].replace(0, np.nan)).round(3)
share.index.name = 'AggAreaCode'
share.to_csv('Output/transit/mode_share_area.csv')

corr = share[share['IsLRT_Corridor'] == 1]
print("corridor areas (IsLRT_Corridor = 1):")
print(f"  car+other {corr['car_other'].sum():,.0f} | bus {corr['bus'].sum():,.0f} | train {corr['train'].sum():,.0f} "
      f"| transit share {(corr['bus'].sum() + corr['train'].sum()) / corr['total'].sum():.1%}")
print(f"all {len(AREAS)} areas: transit share {(share['bus'].sum() + share['train'].sum()) / share['total'].sum():.1%}")
print("\nmode shares by area (origin-side, sorted by transit share):")
print(share.sort_values('transit_share', ascending=False).to_string())

corridor areas (IsLRT_Corridor = 1):
  car+other 105,591 | bus 20,044 | train 218 | transit share 16.1%
all 26 areas: transit share 14.7%

mode shares by area (origin-side, sorted by transit share):
                       AggAreaName  IsLRT_Corridor  car_other     bus  train    total  transit_share
AggAreaCode                                                                                         
12                      Neve Yosef               1      315.0  1601.0    0.0   1916.0          0.836
13                       Hamifrats               1     1154.0  2208.0   69.0   3431.0          0.664
2                            Matam               1     1028.0   705.0   41.0   1774.0          0.421
4                       Neve David               1     2967.0  2160.0    0.0   5127.0          0.421
3                       Neot Peres               1      829.0   302.0    0.0   1131.0          0.267
9                       Lower City               1     2981.0  1026.0   32.0   4039.0         

## Notes

- Train data is **2019** smartcards (pre-COVID), bus is 2022 ticketing, the CAR/OTHER base is THS 2018 — the vintage mix is documented rather than adjusted (see `TRANSIT_DEMAND_PLAN.md` step 5 for the optional alignment).
- Train trips are assigned to the **station's TAZ** — like bus boarding locations, right for corridor loading, more concentrated than door-to-door origins.
- Most rail trips cross the study boundary (the ignored `TAZ 9999` rows plus north↔south/center flows), so the in-sub-area rail contribution is small by construction.
- `ALL_adjusted` mildly overstates transit share by frame (ticketing counts non-residents; the CAR/OTHER base counts residents) — conservative in the right direction for car-to-LRT shift analysis.